In [97]:
import pandas as pd

tweets_df = pd.read_csv('data/post/tweets.csv')
coins_df = pd.read_csv('data/post/main_2.csv')
pd.set_option('display.max_columns', None)

In [98]:
drop_columns = ["observed_at", "delta_seconds", "favorite_count",	"retweet_count",	"reply_count",	"quote_count",	"bookmark_count",	"view_count",	"monitor_flag",	"tweet_last_updated"]
tweets_df.drop(columns=drop_columns, inplace=True)
# select only tweets with coin_address unknown
# tweets_df = tweets_df[tweets_df['coin_address'] == 'unknown']
tweets_df.head()

,observation_id,coin_address,pair_address,tweet_id,tweet_created_at,tweet_text,tweet_sentiment_label,tweet_sentiment_label_prob,tweet_engagement_score,user_id,username,user_description,user_followers_count,user_following_count,verified,is_blue_verified,user_created_at,user_last_updated
0,1,E8CGkGTkVPR41QgbrMyYBMzYWEx5Kajw51x7njSPtfYv,CTgm5FpjkKxg7tv63fGqj2ssJjjShSJ1QsYGXh6NqJAz,1914575339789148558,1745305306,🚨 Trending 🚨\n\n📈 FDV 5MIN 💹 +$20.8K(+31.53%) ...,positive,0.999981,1.659133,4019075112,ノリノスケ@mzdao,you even if you have any more bad manners he w...,274,1139,0,0,1445822118,1745385530
1,2,2Hmkjw9RGSJiMQboj1ULbbWt77xdGvVNKMj6QRCbSbyW,7eErXEaWzJe65AMcBnc6sZ8R7KX7zBpBCaAdcPDBCxN4,1914578022881247694,1745305946,Detect PAID DEXScreener: $SOLUNA 👇\n ...,neutral,0.999959,2.425256,1806913801754963968,Dex signals,Check Paid Dex👇\n https://t.co/Pvons...,7073,32,0,1,1719636806,1745385552
2,3,GsqARQyjNXQxw4R4ASFbVeLu8ZPWBFHqre3YimheBVdD,FC2Jmjse7LUmPYXf7iRVSwCRKi1A5NymJVN9qhxMoK3t,1914583371499626848,1745307221,🤖 FDV Alert 🤖\n\n🌠 - $POG\n💪 - GsqARQyjNXQxw4R...,neutral,0.999875,1.733796,1540809779161219072,TheasGonzalez,Real girls arent perfect Perfect girls arent r...,635,1405,0,0,1656192659,1745386178
3,4,55MLQqMRZ9EKHqcswNxrkhfbt1xPJW4xujRSKTbXUuhd,FHTnZVPX1dGePM9VzgBHV2nkirrkinRnDD6F1Arcj2Am,1914584145285841035,1745307406,📣 Signal &amp; ⚡QUICK TRADE⚡ 👉👉 https://t.co/2...,positive,0.999918,1.830620,1859819270,Vrianfang,Japan Travel Guide | CNN Travel\ndiscord:Jayfa...,297,1191,0,0,1379057624,1745386241
4,5,WVzYFs3mrMRpJ7VMvCApTXG3ZTrfPqkdGBVfR1hpump,FV11DKG7pPDYT1pdhZCRRp2p6ZBSgfCw3GVDAXYZwGVD,1914589022137278884,1745308568,🚨 Trending 🚨\n\n📈 FDV 5MIN 💹 +$1M(+3256.9%) \n...,positive,0.999976,2.028513,1542449247424892928,ClaraasSloan,Whatever is worth doing at all is worth B5A6Y4...,950,1191,0,0,1656583539,1745386344


In [99]:
unknown_count = tweets_df['coin_address'].value_counts()['unknown']

number_of_tweets = tweets_df.shape[0]
known_count = number_of_tweets - unknown_count

print(f"unknown_count: {unknown_count}")
print(f"known_count: {known_count}")

unknown_count: 28141
known_count: 371


In [100]:
coins_dict = {}
pair_dict = {}

for idx, row in coins_df.iterrows():
    coins_dict[row['coin_address']] = row['pair_address']
    pair_dict[row['pair_address']] = row['coin_address']


In [101]:
# regex for finding solana addresses.
import re

# Base-58 alphabet (no 0, O, I, l) • 32–44 chars covers all 32-byte Solana PKs
_SOL_ADDR_RE = re.compile(r"\b[1-9A-HJ-NP-Za-km-z]{32,44}\b")

def list_solana_addresses(text: str) -> list[str]:
    """
    Extract Solana wallet/program addresses from *text*.

    Returns the addresses in first-seen order with duplicates removed.
    """
    return list(dict.fromkeys(_SOL_ADDR_RE.findall(text)))

unique_user_ids = tweets_df['user_id'].unique()
user_ownership = {}
for user_id in unique_user_ids:
    user_tweets = tweets_df[tweets_df['user_id'] == user_id]
    tweets_text = user_tweets['tweet_text'].tolist()
    user_description = user_tweets['user_description'].tolist()[0]

    # user_text = " ".join(tweets_text)

    # user_text = str(user_description) + "\n---\n" + str(user_text)

    # user_text = user_text.lower()

    user_text = str(user_description)

    addresses = list_solana_addresses(user_text)

    if len(addresses) == 1:
    #for address in addresses:
        user_ownership[user_id] = addresses[0]

In [102]:
# get unique user_ids
for idx, row in tweets_df.iterrows():
    if row['coin_address'] == 'unknown':
        tweet = row['tweet_text']
        user_id = row['user_id']
        addresses = list_solana_addresses(tweet)
        if len(addresses) == 1:
            target_address = addresses[0]
            if target_address in coins_dict:
                tweets_df.loc[idx, 'coin_address'] = target_address
                tweets_df.loc[idx, 'pair_address'] = coins_dict[target_address]
            elif target_address in pair_dict:
                tweets_df.loc[idx, 'coin_address'] = pair_dict[target_address]
                tweets_df.loc[idx, 'pair_address'] = target_address
        elif user_id in user_ownership:
            target_address = user_ownership[user_id]
            if target_address in coins_dict:
                tweets_df.loc[idx, 'coin_address'] = target_address
                tweets_df.loc[idx, 'pair_address'] = coins_dict[target_address]
            elif target_address in pair_dict:
                tweets_df.loc[idx, 'coin_address'] = pair_dict[target_address]
                tweets_df.loc[idx, 'pair_address'] = target_address

        



In [103]:
unknown_count = tweets_df['coin_address'].value_counts()['unknown']

number_of_tweets = tweets_df.shape[0]
known_count = number_of_tweets - unknown_count

print(f"unknown_count: {unknown_count}")
print(f"known_count: {known_count}")

unknown_count: 26436
known_count: 2076


In [104]:
known_tweets = tweets_df[tweets_df['coin_address'] != 'unknown']
known_tweets.to_csv('data/post/tweets_resolved.csv', index=False)
tweets_df.to_csv('data/post/tweets_2.csv', index=False)